In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from silver_config import SILVER_TABLES, DEDUP_CONFIG, SILVER_SCHEMA

In [0]:
silver_dfs = [spark.read.table(f"{SILVER_SCHEMA}.{silver_table}") for silver_table in SILVER_TABLES]

In [0]:
all_listings = silver_dfs[0]
for df in silver_dfs[1:]:
    all_listings = all_listings.unionByName(df)

In [0]:
null_price_listings = all_listings_full.filter(F.col("price_amount").isNull())
all_listings = all_listings.filter(F.col("price_amount").isNotNull())

In [0]:
w = Window.partitionBy(*DEDUP_CONFIG["group_keys"][:1]).orderBy("price_amount").rangeBetween(
    -DEDUP_CONFIG["price_range"], DEDUP_CONFIG["price_range"]
)

all_listings = all_listings.withColumn(
    "nearby", F.collect_list(F.struct("listing_id", "description")).over(w)
)

all_listings = all_listings.withColumn(
    "nearby_others",
    F.filter("nearby", lambda x: x["listing_id"] != F.col("listing_id"))
)

all_listings = all_listings.withColumn(
    "distances",
    F.transform("nearby_others", lambda x: F.levenshtein(F.col("description"), x["description"]))
)

all_listings = all_listings.withColumn("min_distance", F.array_min("distances"))

duplicates = all_listings.filter(F.col("min_distance") < 15)

In [0]:
w_rank = Window.partitionBy("location_city", "price_amount").orderBy("listing_id")

duplicates = duplicates.withColumn("dedup_rank", F.row_number().over(w_rank))

urls_to_remove = duplicates.filter(F.col("dedup_rank") != 1).select("source_url")

all_listings = all_listings.join(urls_to_remove, on="source_url", how="left_anti")

In [0]:
all_listings = all_listings.drop(
    "nearby", "nearby_others", "distances", "candidates", "best_match",
    "min_distance", "matched_listing_id", "dedup_rank"
)
all_listings = all_listings.unionByName(null_price_listings)

In [0]:
for table in SILVER_TABLES:
    final_df = all_listings.filter(F.col("source") == table)
    final_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.dedup_{table}")

In [0]:
for table in SILVER_TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {SILVER_SCHEMA}.{table}")
    spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.dedup_{table} RENAME TO {SILVER_SCHEMA}.{table}")